# MEDWATERICE + Random Forest — Google Colab

الدفتر يرفع ملفات 2019 و2020 و2021 وملف `model_integration_dataset.csv`، يفحص Excel، يستخرج قيم الري المرشحة، وينشئ تقارير قابلة للتنزيل. التدريب لا يعمل إلا بعد تأكيد تعريف ووحدة Target؛ الإعداد الافتراضي يمنع استخدام قيم غير محسومة. لا يتم دمج صفوف `model_integration_dataset` مع سجلات التجربة تلقائيًا لأن مستوى البيانات مختلف.

الموديل المقصود يتنبأ بقيمة ري رقمية؛ لذلك المقاييس الصحيحة MAE وRMSE وR² وليست accuracy تصنيفية.


## 1. تثبيت الحزم واستيرادها

In [1]:
import importlib.util, subprocess, sys
needed = {"pandas":"pandas", "numpy":"numpy", "openpyxl":"openpyxl", "sklearn":"scikit-learn", "joblib":"joblib"}
missing = [pkg for module, pkg in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from pathlib import Path
from io import BytesIO
import zipfile, re, json, warnings
import numpy as np
import pandas as pd
import joblib
from IPython.display import display
from google.colab import files
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import TransformedTargetRegressor
print("تم تجهيز المكتبات. هذا القسم لا يدرّب موديلًا.")

تم تجهيز المكتبات. هذا القسم لا يدرّب موديلًا.


## 2. رفع الملفات الأربعة

In [3]:
uploaded = files.upload()
zip_uploads = {name: data for name, data in uploaded.items() if name.lower().endswith(".zip")}
csv_uploads = {name: data for name, data in uploaded.items() if name.lower().endswith(".csv")}
print("ZIP:", list(zip_uploads))
print("CSV:", list(csv_uploads))
if len(zip_uploads) != 3:
    raise ValueError("ارفعي ملفات MEDWATERICE ZIP الثلاثة معًا.")
if not csv_uploads:
    raise ValueError("ارفعي model_integration_dataset.csv أيضًا.")


Saving doi-10.13130-rd_unimi-ffenin.zip to doi-10.13130-rd_unimi-ffenin (1).zip
Saving doi-10.13130-rd_unimi-bdyjey.zip to doi-10.13130-rd_unimi-bdyjey.zip
Saving doi-10.13130-rd_unimi-0zxbcn.zip to doi-10.13130-rd_unimi-0zxbcn.zip
Saving model_integration_dataset.csv to model_integration_dataset.csv
ZIP: ['doi-10.13130-rd_unimi-ffenin (1).zip', 'doi-10.13130-rd_unimi-bdyjey.zip', 'doi-10.13130-rd_unimi-0zxbcn.zip']
CSV: ['model_integration_dataset.csv']


## 3. فك ملفات ZIP بأمان وتحديد سنة كل ملف

In [4]:
YEAR_TOKENS = {"0zxbcn": 2019, "bdyjey": 2020, "ffenin": 2021}
ROOT = Path("/content/medwaterice_uploads")
ROOT.mkdir(parents=True, exist_ok=True)
workbooks = []
archive_manifest = []
for name, payload in zip_uploads.items():
    lower = name.lower()
    year = next((yr for token, yr in YEAR_TOKENS.items() if token in lower), None)
    if year is None:
        raise ValueError(f"لم أتعرف على سنة الأرشيف: {name}")
    dest_root = ROOT / str(year)
    dest_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(BytesIO(payload)) as zf:
        base = dest_root.resolve()
        for info in zf.infolist():
            dest = (dest_root / info.filename).resolve()
            if dest != base and base not in dest.parents:
                raise ValueError(f"مسار غير آمن داخل ZIP: {info.filename}")
        zf.extractall(dest_root)
    for p in dest_root.rglob("*"):
        if p.is_file():
            archive_manifest.append({"year":year,"archive":name,"filename":p.name,"bytes":p.stat().st_size})
            if p.suffix.lower()==".xlsx": workbooks.append((year,p))
print("Excel workbooks:",len(workbooks))
display(pd.DataFrame(archive_manifest).groupby(["year", "filename"]).size().groupby(level=0).size().rename("files"))


Excel workbooks: 14


,files
year,
2019,6
2020,10
2021,10


## 4. قراءة ملف integration للفحص فقط

In [5]:
integration_name = next((n for n in csv_uploads if "model_integration_dataset" in n.lower()), None)
if integration_name is None:
    raise ValueError("اسم ملف CSV لازم يتضمن model_integration_dataset.")
INTEGRATION = pd.read_csv(BytesIO(csv_uploads[integration_name]), low_memory=False)
print("Integration shape:", INTEGRATION.shape)
print("Exact duplicate rows:", int(INTEGRATION.duplicated().sum()))
display(pd.DataFrame({"dtype":INTEGRATION.dtypes.astype(str),"missing":INTEGRATION.isna().sum()}).sort_values("missing",ascending=False).head(25))
for c in ["actual_irrigation_status", "main_model_status"]:
    if c in INTEGRATION:
        display(INTEGRATION[c].fillna("<missing>").value_counts().rename_axis(c).to_frame("rows"))
print("مهم: لم ندمج هذا الملف مع بيانات MEDWATERICE؛ نحتاج مفتاح موقع/قطعة وفترة متوافقًا أولًا.")


Integration shape: (266, 68)
Exact duplicate rows: 0


,dtype,missing
organic_carbon,float64,14
bulk,float64,14
silt_pct,float64,14
clay_pct,float64,14
sand_pct,float64,14
bottom_depth_cm,float64,14
smu_id,float64,14
awc_component_status,object,14
ph_water,float64,14
texture_usda,float64,14


,rows
actual_irrigation_status,
NOT_FIELD_MEASURED,266


,rows
main_model_status,
FAO_BASED_SCENARIO_NOT_FIELD_VALIDATED,266


مهم: لم ندمج هذا الملف مع بيانات MEDWATERICE؛ نحتاج مفتاح موقع/قطعة وفترة متوافقًا أولًا.


## 5. جرد كل أوراق Excel

In [7]:
sheet_rows=[]
for year, path in workbooks:
    try:
        book = pd.ExcelFile(path)
        for sheet in book.sheet_names:
            frame = pd.read_excel(path, sheet_name=sheet, header=None, dtype=object)
            nonempty = frame.dropna(how="all").dropna(axis=1,how="all")
            sheet_rows.append({"year":year,"workbook":path.name,"sheet":sheet,
                "nonempty_rows":len(nonempty),"used_columns":nonempty.shape[1],
                "missing_cells_used_range":int(nonempty.isna().sum().sum()),
                "duplicate_nonempty_rows":int(nonempty.duplicated().sum()) if len(nonempty) else 0})
    except Exception as e:
        sheet_rows.append({"year":year,"workbook":path.name,"sheet":"READ_ERROR","error":str(e)})
SHEET_AUDIT=pd.DataFrame(sheet_rows)
print("Audited sheet rows:",len(SHEET_AUDIT))
display(SHEET_AUDIT.groupby("year").agg(workbooks=("workbook","nunique"),sheets=("sheet","size")))
display(SHEET_AUDIT.groupby("sheet").agg(workbooks=("workbook","nunique"),median_nonempty_rows=("nonempty_rows","median")))


Audited sheet rows: 182


,workbooks,sheets
year,,
2019,2,26
2020,6,78
2021,6,78


,workbooks,median_nonempty_rows
sheet,,
Agro-climatic,14,186.0
Crop evol,14,2.0
Gas emiss,14,2.0
General,14,113.0
Groundwater,14,2.0
Irrig,14,4.0
Managm oper,14,103.0
Other water,14,18.0
Soil phy-ch,14,3.0


## 6. استخراج قيم Inflow المرشحة من أوراق Irrig

In [8]:
def norm_text(x):
    return re.sub(r"\s+", " ", str(x)).strip().lower()

def workbook_labels(name):
    s=name.upper()
    strategy=("Hybrid" if "HYBRID" in s else "Sprinkler" if "SPRINKLER" in s else "Drip" if "DRIP" in s else "WFL" if "WFL" in s else "unknown")
    quality=("freshwater" if "FRESHWATER" in s else "drainage" if "DRAINAGE" in s else "unspecified")
    return strategy,quality

candidate_rows=[]
for year,path in workbooks:
    try:
        raw=pd.read_excel(path,sheet_name="Irrig",header=None,dtype=object)
        header_i=None; inflow_i=None; outflow_i=None; method_i=None; date_i=None; das_i=None
        for i,row in raw.iterrows():
            labels=[norm_text(v) for v in row.tolist()]
            if any("irrig inflow" in x for x in labels):
                header_i=i
                inflow_i=next(j for j,x in enumerate(labels) if "irrig inflow" in x)
                outflow_i=next((j for j,x in enumerate(labels) if "irrig outflow" in x),None)
                method_i=next((j for j,x in enumerate(labels) if "cultivation method" in x),None)
                date_i=next((j for j,x in enumerate(labels) if x=="date"),None)
                das_i=next((j for j,x in enumerate(labels) if "days after sowing" in x),None)
                break
        if header_i is None: continue
        inflow_header=str(raw.iat[header_i,inflow_i])
        meta=pd.read_excel(path,sheet_name="General",header=None,dtype=object)
        meta_unit=meta_flag=meta_method=None
        for _,row in meta.iterrows():
            vals=[norm_text(v) for v in row.tolist()]
            if any("irrigation inflow"==v for v in vals):
                j=next(k for k,v in enumerate(vals) if v=="irrigation inflow")
                meta_unit=row.iloc[j+1] if j+1<len(row) else None
                meta_flag=row.iloc[j+2] if j+2<len(row) else None
                meta_method=row.iloc[j+5] if j+5<len(row) else None
                break
        strategy,quality=workbook_labels(path.name)
        for r in range(header_i+1,len(raw)):
            value=pd.to_numeric(str(raw.iat[r,inflow_i]).replace(",",""),errors="coerce")
            if pd.isna(value): continue
            method=raw.iat[r,method_i] if method_i is not None else None
            date=raw.iat[r,date_i] if date_i is not None else None
            das=raw.iat[r,das_i] if das_i is not None else None
            outflow=raw.iat[r,outflow_i] if outflow_i is not None else None
            group_method=str(method).strip() if pd.notna(method) else "not-specified"
            candidate_rows.append({"year":year,"workbook":path.name,"site":"Sakha (header/source workbook)",
                "irrigation_strategy":strategy,"water_quality":quality,"cultivation_method":method,
                "target_value":float(value),"target_unit_from_Irrig_header":inflow_header,
                "metadata_unit_from_General":meta_unit,"measurement_flag_from_General":meta_flag,
                "measurement_method_from_General":meta_method,"date":date,"days_after_sowing":das,
                "outflow_value":outflow,
                # Same-year/same-treatment fresh/drainage rows stay in one validation group.
                "experiment_group_id":f"{year}|{strategy}|{group_method}"})
    except Exception as e:
        print("Read issue:",path.name,e)
CANDIDATES=pd.DataFrame(candidate_rows)
if CANDIDATES.empty: raise ValueError("لم أستخرج أي قيم Inflow؛ راجعي أسماء أوراق Irrig.")
print("Candidate values:",len(CANDIDATES),"; independent treatment groups:",CANDIDATES.experiment_group_id.nunique())
display(CANDIDATES)


Candidate values: 20 ; independent treatment groups: 12


,year,workbook,site,irrigation_strategy,water_quality,cultivation_method,target_value,target_unit_from_Irrig_header,metadata_unit_from_General,measurement_flag_from_General,measurement_method_from_General,date,days_after_sowing,outflow_value,experiment_group_id
0,2021,MEDWATERICE DMP_CS6_EGYPT_HYBRID-DRAINAGE WATE...,Sakha (header/source workbook),Hybrid,drainage,None,10559.00,Irrig inflow (m3/ha)\nSakha,m3/h,M,Mechanic and Ultrasonic water flowmeters at th...,NaN,NaN,NaN,2021|Hybrid|not-specified
1,2021,MEDWATERICE DMP_CS6_EGYPT_SPRINKLER_2021.xlsx,Sakha (header/source workbook),Sprinkler,unspecified,None,9102.00,Irrig inflow (m3/ha)\nSakha,m3/h,M,Mechanic and Ultrasonic water flowmeters at th...,NaN,NaN,NaN,2021|Sprinkler|not-specified
2,2021,MEDWATERICE DMP_CS6_EGYPT_HYBRID-FRESHWATER_20...,Sakha (header/source workbook),Hybrid,freshwater,None,10559.00,Irrig inflow (m3/ha)\nSakha,m3/h,M,Mechanic and Ultrasonic water flowmeters at th...,NaN,NaN,NaN,2021|Hybrid|not-specified
3,2021,MEDWATERICE DMP_CS6_EGYPT_DRIP_2021.xlsx,Sakha (header/source workbook),Drip,unspecified,None,7370.00,Irrig inflow (m3/ha)\nSakha,m3/h,M,Mechanic and Ultrasonic water flowmeters at th...,NaN,NaN,NaN,2021|Drip|not-specified
4,2021,MEDWATERICE DMP_CS6_EGYPT_WFL-DRAINAGE WATER_2...,Sakha (header/source workbook),WFL,drainage,None,14585.00,Irrig inflow (m3/ha)\nSakha,m3/h,M,Mechanic and Ultrasonic water flowmeters at th...,NaN,NaN,NaN,2021|WFL|not-specified
5,2021,MEDWATERICE DMP_CS6_EGYPT_WFL-FRESHWATER_2021....,Sakha (header/source workbook),WFL,freshwater,None,14585.00,Irrig inflow (m3/ha)\nSakha,m3/h,M,Mechanic and Ultrasonic water flowmeters at th...,NaN,NaN,NaN,2021|WFL|not-specified
6,2020,MEDWATERICE DMP_CS6_EGYPT_WFL-DRAINAGE WATER_2...,Sakha (header/source workbook),WFL,drainage,None,14937.00,Irrig inflow (m3/ha)\nSakha,m3/h,M,Mechanic and Ultrasonic water flowmeters at th...,NaN,NaN,NaN,2020|WFL|not-specified
7,2020,MEDWATERICE DMP_CS6_EGYPT_HYBRID-DRAINAGE WATE...,Sakha (header/source workbook),Hybrid,drainage,None,11216.94,Irrig inflow (m3/ha)\nSakha,m3/h,M,Mechanic and Ultrasonic water flowmeters at th...,NaN,NaN,NaN,2020|Hybrid|not-specified
8,2020,MEDWATERICE DMP_CS6_EGYPT_WFL-FRESHWATER_2020....,Sakha (header/source workbook),WFL,freshwater,None,14937.00,Irrig inflow (m3/ha)\nSakha,m3/h,M,Mechanic and Ultrasonic water flowmeters at th...,NaN,NaN,NaN,2020|WFL|not-specified
9,2020,MEDWATERICE DMP_CS6_EGYPT_DRIP_2020.xlsx,Sakha (header/source workbook),Drip,unspecified,None,7894.00,Irrig inflow (m3/ha)\nSakha,m3/h,M,Mechanic and Ultrasonic water flowmeters at th...,NaN,NaN,NaN,2020|Drip|not-specified


## 7. فحص التكرار والوحدات والمفاتيح — من غير حذف أو دمج

In [9]:
print("Duplicate full rows:",int(CANDIDATES.duplicated().sum()))
print("Rows missing date:",int(CANDIDATES.date.isna().sum()),"/",len(CANDIDATES))
print("Rows missing outflow:",int(CANDIDATES.outflow_value.isna().sum()),"/",len(CANDIDATES))
display(CANDIDATES.groupby(["year","irrigation_strategy","water_quality"],dropna=False).agg(
    rows=("target_value","size"),values=("target_value",lambda s:sorted(s.dropna().unique().tolist()))))
print("Units in Inflow header:",CANDIDATES.target_unit_from_Irrig_header.value_counts(dropna=False).to_dict())
print("Units in General metadata:",CANDIDATES.metadata_unit_from_General.value_counts(dropna=False).to_dict())
print("المصدر يصف inflow كقياس M بالعداد، لكن m3/h مقابل m3/ha يحتاج حسمًا قبل اعتبار الرقم Target نهائيًا.")


Duplicate full rows: 0
Rows missing date: 20 / 20
Rows missing outflow: 20 / 20


rows                       values
year irrigation_strategy water_quality                                   
2019 WFL                 drainage          4  [10115.0, 10234.0, 11305.0]
                         freshwater        4  [10115.0, 10234.0, 11305.0]
2020 Drip                unspecified       1                     [7894.0]
     Hybrid              drainage          1                   [11216.94]
                         freshwater        1                   [11216.94]
     Sprinkler           unspecified       1                     [9487.0]
     WFL                 drainage          1                    [14937.0]
                         freshwater        1                    [14937.0]
2021 Drip                unspecified       1                     [7370.0]
     Hybrid              drainage          1                    [10559.0]
                         freshwater        1                    [10559.0]
     Sprinkler           unspecified       1                     [9102.0]
     WFL                 drainage          1                    [14585.0]
                         freshwater        1                    [14585.0]

Units in Inflow header: {'Irrig inflow (m3/ha)\nSakha': 20}
Units in General metadata: {'m3/h': 20}
المصدر يصف inflow كقياس M بالعداد، لكن m3/h مقابل m3/ha يحتاج حسمًا قبل اعتبار الرقم Target نهائيًا.


## 8. تنظيف غير هدّام وتصدير تقارير الفحص

In [10]:
# لا نغيّر CANDIDATES أو الملفات الخام، ولا نحذف الصفوف المكررة تلقائيًا.
CANDIDATE_PROFILE=pd.DataFrame({"dtype":CANDIDATES.dtypes.astype(str),"missing":CANDIDATES.isna().sum(),
    "missing_pct":(100*CANDIDATES.isna().mean()).round(1)})
display(CANDIDATE_PROFILE)
OUT=Path("/content/colab_outputs"); OUT.mkdir(exist_ok=True)
CANDIDATES.to_csv(OUT/"MEDWATERICE_target_candidates.csv",index=False,encoding="utf-8-sig")
SHEET_AUDIT.to_csv(OUT/"MEDWATERICE_sheet_inventory.csv",index=False,encoding="utf-8-sig")
INTEGRATION.isna().sum().rename("missing_cells").to_csv(OUT/"integration_missing_profile.csv",encoding="utf-8-sig")
print("Saved audit CSVs to",OUT)


,dtype,missing,missing_pct
year,int64,0,0.0
workbook,object,0,0.0
site,object,0,0.0
irrigation_strategy,object,0,0.0
water_quality,object,0,0.0
cultivation_method,object,12,60.0
target_value,float64,0,0.0
target_unit_from_Irrig_header,object,0,0.0
metadata_unit_from_General,object,0,0.0
measurement_flag_from_General,object,0,0.0


Saved audit CSVs to /content/colab_outputs


## 9. اعتماد تعريف Target والخصائص قبل التدريب

القيم في `Irrig` مرشحة لتكون إجماليات مياه التجربة. افتراضيًا التدريب مغلق إلى أن يؤكد مصدر البيانات هل الوحدة النهائية فعلًا m3/ha وما إذا كانت القيمة إجمالي موسم/مساحة. لا تفعّلي التأكيد بالتخمين. ملف integration لا يوفّر target مقاسًا؛ لذلك لن نلحق صفوفه بقياسات MEDWATERICE. بعد التحقق، يمكن تدريب نموذج تجريبي للتنبؤ بقيم محطة التجربة، وليس تعميمه تلقائيًا على مزارع أخرى.

In [11]:
TARGET="target_value"
TARGET_UNIT_CONFIRMED=False  # غيّريها فقط بعد تأكيد موثق أن قيم Irrig هي m3/ha وبأي فترة/مساحة.
RESEARCH_STATION_TARGET_APPROVED=False  # يلزم اعتماد صريح لنطاق التنبؤ: تجارب MEDWATERICE فقط.
PREDICTION_FEATURES=["year","irrigation_strategy","water_quality","cultivation_method"]
GROUP="experiment_group_id"

gate_reasons=[]
if not TARGET_UNIT_CONFIRMED: gate_reasons.append("وحدة/تعريف Target غير محسوم (General=m3/h وIrrig=m3/ha).")
if not RESEARCH_STATION_TARGET_APPROVED: gate_reasons.append("نطاق الهدف التجريبي للمحطة لم يُعتمد بعد.")
if CANDIDATES[TARGET].isna().any(): gate_reasons.append("توجد قيم Target مفقودة؛ لا نملأ Target بالتخمين.")
if CANDIDATES[GROUP].nunique()<10: gate_reasons.append("عدد مجموعات التجربة المستقلة أقل من 10.")
if any(c not in CANDIDATES for c in PREDICTION_FEATURES): gate_reasons.append("بعض الخصائص المطلوبة غير موجودة.")
if (pd.to_numeric(CANDIDATES[TARGET],errors="coerce")<0).any(): gate_reasons.append("توجد Target سالبة تحتاج مراجعة مصدرها.")
TRAINING_ALLOWED=(len(gate_reasons)==0)
print("TRAINING_ALLOWED =",TRAINING_ALLOWED)
if gate_reasons:
    print("أسباب عدم التدريب:")
    for reason in gate_reasons: print("-",reason)
else:
    print("التحقق الأساسي اجتاز؛ سيظل التقييم محدودًا بصغر العينة وثلاث سنوات فقط.")


TRAINING_ALLOWED = False
أسباب عدم التدريب:
- وحدة/تعريف Target غير محسوم (General=m3/h وIrrig=m3/ha).
- نطاق الهدف التجريبي للمحطة لم يُعتمد بعد.


## 10. Random Forest حتى 1000 شجرة مع إيقاف مبكر

In [12]:
if not TRAINING_ALLOWED:
    print("التدريب متوقف لحماية صحة النتيجة. لا تغيّري أعلام الاعتماد قبل توثيق القياس والوحدة.")
else:
    model_data=CANDIDATES[PREDICTION_FEATURES+[TARGET,GROUP]].copy()
    model_data[TARGET]=pd.to_numeric(model_data[TARGET],errors="coerce")
    model_data=model_data.dropna(subset=[TARGET,GROUP])
    X=model_data[PREDICTION_FEATURES]
    y=model_data[TARGET]
    groups=model_data[GROUP]
    numeric=[c for c in PREDICTION_FEATURES if pd.api.types.is_numeric_dtype(X[c])]
    categorical=[c for c in PREDICTION_FEATURES if c not in numeric]
    prep=ColumnTransformer([
        ("num",Pipeline([("impute",SimpleImputer(strategy="median"))]),numeric),
        ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),
                         ("encode",OneHotEncoder(handle_unknown="ignore"))]),categorical)
    ],remainder="drop")

    # Hold out complete treatment groups. A separate group validation set chooses tree count.
    outer=GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=42)
    dev_idx,test_idx=next(outer.split(X,y,groups))
    X_dev,y_dev,g_dev=X.iloc[dev_idx],y.iloc[dev_idx],groups.iloc[dev_idx]
    X_test,y_test=X.iloc[test_idx],y.iloc[test_idx]
    inner=GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=17)
    tr_idx,val_idx=next(inner.split(X_dev,y_dev,g_dev))
    X_tr,y_tr=X_dev.iloc[tr_idx],y_dev.iloc[tr_idx]
    X_val,y_val=X_dev.iloc[val_idx],y_dev.iloc[val_idx]
    prep_early=prep.fit(X_tr)
    X_tr_t=prep_early.transform(X_tr); X_val_t=prep_early.transform(X_val)
    forest=RandomForestRegressor(n_estimators=50,warm_start=True,random_state=42,n_jobs=-1,
                                 min_samples_leaf=2,max_features=1.0)
    best_rmse=float("inf"); best_n=50; patience=0; history=[]
    for n_trees in range(50,1001,50):
        forest.set_params(n_estimators=n_trees)
        forest.fit(X_tr_t,y_tr)
        pred=forest.predict(X_val_t)
        rmse=float(np.sqrt(mean_squared_error(y_val,pred)))
        history.append({"trees":n_trees,"validation_RMSE":rmse})
        if rmse < best_rmse-1e-9:
            best_rmse=rmse; best_n=n_trees; patience=0
        else:
            patience+=1
        if patience>=4: break
    display(pd.DataFrame(history))
    print("Selected tree count:",best_n,"(max 1000; early stop after 4 non-improving steps)")

    # Refit on all development groups; keep test groups untouched until final evaluation.
    final_prep=prep.fit(X_dev)
    X_dev_t=final_prep.transform(X_dev); X_test_t=final_prep.transform(X_test)
    final_model=RandomForestRegressor(n_estimators=best_n,random_state=42,n_jobs=-1,
                                      min_samples_leaf=2,max_features=1.0)
    final_model.fit(X_dev_t,y_dev)
    test_pred=final_model.predict(X_test_t)
    metrics={"MAE":mean_absolute_error(y_test,test_pred),
             "RMSE":float(np.sqrt(mean_squared_error(y_test,test_pred))),
             "R2":r2_score(y_test,test_pred) if len(y_test)>1 else np.nan,
             "test_rows":len(y_test),"test_groups":int(groups.iloc[test_idx].nunique()),
             "selected_trees":best_n}
    display(pd.Series(metrics,name="holdout result"))
    print("هذه نتيجة تجريبية لمحطة البحث والبيانات المتاحة، وليست ضمانًا لدقة مزارع أخرى.")


التدريب متوقف لحماية صحة النتيجة. لا تغيّري أعلام الاعتماد قبل توثيق القياس والوحدة.


## 11. حفظ الموديل والتقارير وتنزيلها

In [13]:
if TRAINING_ALLOWED:
    import pickle
    bundle={"preprocessor":final_prep,"model":final_model,"features":PREDICTION_FEATURES,
            "target":TARGET,"target_unit":"m3/ha (only if source confirmation is true)",
            "scope":"MEDWATERICE research-station treatment prototype","metrics":metrics}
    model_file=OUT/"medwaterice_random_forest.joblib"
    metrics_file=OUT/"medwaterice_holdout_metrics.csv"
    joblib.dump(bundle,model_file)
    pd.DataFrame([metrics]).to_csv(metrics_file,index=False)
    files.download(str(model_file)); files.download(str(metrics_file))
else:
    print("لم يُحفظ موديل لأن Target لم يعتمد. ملفات التدقيق جاهزة للتنزيل في الخلية التالية.")


لم يُحفظ موديل لأن Target لم يعتمد. ملفات التدقيق جاهزة للتنزيل في الخلية التالية.


## 12. تنزيل تقارير الفحص

In [15]:
for p in [OUT/"MEDWATERICE_target_candidates.csv",OUT/"MEDWATERICE_sheet_inventory.csv",OUT/"integration_missing_profile.csv"]:
    if p.exists(): files.download(str(p))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>